# Soup Kernel Challenge — Мерей: pinned Kaggle T4 working copy

This notebook clones `https://github.com/promaladoi/Decentrathon-6.0/Мерей` at code commit `b7c406dea7dbdc379149910efe044dccf155d083`,
then runs from `/kaggle/working/repo/Мерей`. Set **GPU T4 x2** and **Internet on**.
It verifies the measured solution SHA256 before tests, then runs the unchanged
harness with two base and two solution runs. Keep both GPUs idle during measurement.

Recorded full development version: [Kaggle 363837033](https://www.kaggle.com/code/promaladoi/soup-kaggle-development?scriptVersionId=363837033).
That saved version used the immutable embedded DDC snapshot. This clone-based
notebook is prepared for reproduction and has not itself been executed.
The original platform submission template remains `templates/kaggle_notebook.ipynb`.
Final submission still requires its platform repository commit and a saved public run.
Use optional read-only `GITHUB_TOKEN` and `HF_TOKEN` through Kaggle Secrets.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "b7c406dea7dbdc379149910efe044dccf155d083"
EXPECTED_SOLUTION_SHA256 = "698b33301c9a191f3b8bfadc502a0c15d7a08870579c465cfed119a06a2d4d77"


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
# Remove an incompatible optional package before all tests and measurements.
!python -m pip uninstall -y torchao
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
%cd /kaggle/working/repo/Мерей
import hashlib, os, subprocess, sys
from pathlib import Path
digest = hashlib.sha256()
for path in sorted(Path("solution").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        digest.update(str(path.relative_to("solution")).encode())
        digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOLUTION_SHA256, "Clone differs from measured solution bytes"
sys.path.insert(0, str(Path("harness").resolve()))
from common import ENV
def run_repo(arguments, **overrides):
    environment = dict(ENV)
    environment.update(overrides)
    subprocess.run([sys.executable, *arguments], env=environment, check=True)
run_repo(["task/prepare_data.py"])
run_repo(["-m", "pytest", "solution", "-q"])
run_repo(["-m", "pytest", "tests", "-q"], SOUP_APPLY_SOLUTION="1")


In [ ]:
run_repo(["harness/run_task.py", "--repeats", "2"])


In [ ]:
run_repo(["harness/check_rules.py"])
run_repo(["harness/score.py", "runs/result.json"])


While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.